# Phase 5 & 6 – Multimodal RAG Notebook

Explores:
- Table extraction and retrieval
- Image captioning and retrieval
- ColPali-style visual page retrieval
- Comparison: text-only vs multimodal answers

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

## Phase 5.1 – Table Extraction

In [ ]:
from src.ingestion.extract_tables import extract_tables_from_pdf, save_extracted_tables

tables = extract_tables_from_pdf()
save_extracted_tables(tables)

print(f'Total tables found: {len(tables)}')

# Show page distribution
pages_with_tables = [t['page_number'] for t in tables]
plt.figure(figsize=(12, 3))
plt.scatter(pages_with_tables, [1]*len(pages_with_tables), marker='|', s=200, color='steelblue')
plt.yticks([])
plt.xlabel('Page Number')
plt.title(f'Pages Containing Tables ({len(tables)} total)')
plt.tight_layout()
plt.show()

In [ ]:
# Preview a table
print('--- Table Preview (Page', tables[0]['page_number'], ') ---')
print(tables[0]['markdown'][:600])

## Phase 5.2 – Image Extraction & Captioning

In [ ]:
from src.ingestion.extract_images import extract_images_from_pdf, save_extracted_images

# caption=False for a quick first run (no Gemini API calls)
# Set caption=True to get Gemini descriptions
images = extract_images_from_pdf(caption=False)
save_extracted_images(images)

print(f'Total images extracted: {len(images)}')

In [ ]:
# Display some extracted images
valid_images = [img for img in images if Path(img['file_path']).exists()]

if valid_images:
    fig, axes = plt.subplots(1, min(4, len(valid_images)), figsize=(16, 4))
    for ax, img_meta in zip(axes, valid_images[:4]):
        try:
            pil_img = Image.open(img_meta['file_path'])
            ax.imshow(pil_img)
            ax.set_title(f"Page {img_meta['page_number']}\n{img_meta['width']}x{img_meta['height']}", fontsize=8)
            ax.axis('off')
        except Exception as e:
            ax.set_title(f'Error: {e}')
    plt.tight_layout()
    plt.show()
else:
    print('No image files found on disk. Run extract_images_from_pdf() first.')

## Multimodal Chunk Distribution

In [ ]:
from src.retrieval.multimodal_retriever import load_all_chunks_multimodal

all_chunks = load_all_chunks_multimodal()

# Count by type
type_counts = {}
for c in all_chunks:
    ct = c.get('content_type', 'unknown')
    type_counts[ct] = type_counts.get(ct, 0) + 1

print('Chunk distribution:')
for k, v in type_counts.items():
    print(f'  {k:<10} {v:>5} chunks  ({100*v/len(all_chunks):.1f}%)')

plt.pie(type_counts.values(), labels=type_counts.keys(),
        autopct='%1.0f%%', colors=['#2196F3', '#4CAF50', '#FF9800'])
plt.title(f'Multimodal Chunk Distribution (total: {len(all_chunks)})')
plt.show()

## Test Multimodal vs Text-only RAG

In [ ]:
from src.rag_pipeline import RAGPipeline
from src.ingestion.chunk_text import load_chunks

# Phase 1: text only
text_chunks    = load_chunks()
text_pipeline  = RAGPipeline('faiss').load(chunks=text_chunks)

# Phase 5: all modalities
# NOTE: You need to run run_phase5_multimodal.py first to rebuild the index
# multi_pipeline = RAGPipeline('faiss').load(chunks=all_chunks)

table_query = 'What was IFC net income according to the financial statements table?'

print(f'Query: {table_query}\n')

# Text-only answer
print('=== Text-only RAG Answer ===')
text_chunks_retrieved = text_pipeline.get_context(table_query, top_k=3)
types_retrieved = [c['content_type'] for c in text_chunks_retrieved]
print(f'Retrieved types: {types_retrieved}')
ans_text = text_pipeline.query(table_query)
print(ans_text[:400])

## Phase 6 – ColPali Visual Page Preview

In [ ]:
# Show rasterised pages (if Phase 6 has been run)
pages_dir = Path('../data/processed/images/pages')

if pages_dir.exists():
    page_files = sorted(pages_dir.glob('page_*.png'))[:6]
    if page_files:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        for ax, pf in zip(axes.flat, page_files):
            img = Image.open(pf)
            ax.imshow(img)
            ax.set_title(pf.stem, fontsize=9)
            ax.axis('off')
        plt.suptitle('Rasterised PDF Pages for ColPali-style Retrieval', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print('No rasterised pages found. Run scripts/run_phase6_colpali.py first.')
else:
    print('Pages directory not found. Run Phase 6 script first.')

In [ ]:
# Load and display page descriptions (if Phase 6 has been run)
desc_file = Path('../data/processed/images/page_descriptions.json')

if desc_file.exists():
    with open(desc_file) as f:
        descriptions = json.load(f)
    
    print(f'Total page descriptions: {len(descriptions)}')
    print('\n--- Sample Description (Page 1) ---')
    print(descriptions[0]['description'][:500])
else:
    print('No page descriptions found. Run scripts/run_phase6_colpali.py first.')